
# Experiment: Cross-Experiment Error Analysis Deep Dive

Objective:
- Go beyond the first-pass failure labels and ask what *kind* of wrong answers the models produce across experiments.
- Separate subtle, target-language near-misses from harder failures like source copying, script failure, decoding collapse, and English task escape.
- Quantify how those families move with difficulty (`depth`, grammar size, agreement condition, target orthography) and show concrete examples of each.

Success criteria:
- A compact taxonomy that explains most wrong outputs.
- At least one figure that shows cross-experiment error-family shifts.
- Conditioned slices for `agreement` and `orthography`, plus representative examples.


In [ ]:
from __future__ import annotations

import importlib
import json
import re
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

try:
    from wordfreq import zipf_frequency
except ImportError:
    zipf_frequency = None

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.precision", 3)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".project-root").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root via .project-root")


PROJECT_ROOT = find_project_root(Path.cwd())
CACHE_DIR = PROJECT_ROOT / "notebooks" / "cache" / "error-analysis"
DATA_DIR = PROJECT_ROOT / "data"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

aes = importlib.import_module("aesthetics")
FIG_HEIGHT_DOUBLEROW_DIFFAXES_IN = aes.FIG_HEIGHT_DOUBLEROW_DIFFAXES_IN

FIGSIZE_STACKED = (aes.COLM_PAPER_WIDTH_IN, 2.2 * FIG_HEIGHT_DOUBLEROW_DIFFAXES_IN)
FIGSIZE_HEATMAP = (
    aes.COLM_PAPER_WIDTH_IN,
    1.5 * FIG_HEIGHT_DOUBLEROW_DIFFAXES_IN,
)
FIGSIZE_TWO_PANEL = (aes.COLM_PAPER_WIDTH_IN, 1.4 * FIG_HEIGHT_DOUBLEROW_DIFFAXES_IN)
FIGSIZE_THREE_PANEL = (
    aes.COLM_PAPER_WIDTH_IN,
    1.45 * FIG_HEIGHT_DOUBLEROW_DIFFAXES_IN,
)
FIG_SIZE_SINGLE = (aes.COLM_PAPER_WIDTH_IN, aes.FIG_HEIGHT_SINGLEROW_IN)

WORD_ORDER_EXP = "wordorder_large_exp"
ORTHOGRAPHY_EXP = "orthography_large_exp"
WORD_ORDER_DATASETS = ("wordorder_large_exp",)
ORTHOGRAPHY_DATASETS = ("orthography_large_exp",)

if WORD_ORDER_EXP not in WORD_ORDER_DATASETS:
    raise ValueError(
        f"WORD_ORDER_EXP must be one of {WORD_ORDER_DATASETS}, got {WORD_ORDER_EXP!r}"
    )
if ORTHOGRAPHY_EXP not in ORTHOGRAPHY_DATASETS:
    raise ValueError(
        f"ORTHOGRAPHY_EXP must be one of {ORTHOGRAPHY_DATASETS}, "
        f"got {ORTHOGRAPHY_EXP!r}"
    )

EXPERIMENT_ORDER = ["size", "wordorder", "agreement", "orthography"]
MODEL_ORDER = [
    model
    for model in aes.MODEL_ORDER
    if model.startswith("gpt-5") or "gemma-3" in model
]
FAMILY_ORDER = [
    "pure_reordering",
    "licensed_near_miss",
    "truncation_or_omission",
    "source_leakage",
    "script_or_form",
    "unlicensed_or_hallucinated",
    "task_escape",
    "degeneration",
    "other",
]
FAMILY_PALETTE = {
    "pure_reordering": "#5B8FF9",
    "licensed_near_miss": "#2A9D8F",
    "truncation_or_omission": "#F4A261",
    "source_leakage": "#E76F51",
    "script_or_form": "#8AB17D",
    "task_escape": "#7A3B69",
    "degeneration": "#6C757D",
    "unlicensed_or_hallucinated": "#B08968",
    "other": "#CED4DA",
}
FAMILY_LABELS = {
    "pure_reordering": "word ordering",
    "licensed_near_miss": "recall error",
    "truncation_or_omission": "omission",
    "source_leakage": "source vocab",
    "script_or_form": "orthography",
    "task_escape": "task escape",
    "degeneration": "degeneration",
    "unlicensed_or_hallucinated": "hallucinated vocab",
    "other": "other",
}
FAMILY_PALETTE_LABELS = {
    FAMILY_LABELS[family]: color for family, color in FAMILY_PALETTE.items()
}
META_RE = re.compile(
    (
        r"(?:translate|translation|cannot|cant|can not|unable|sorry|input|"
        r"source|sentence|grammar|scfg|exact|determine|provided|given|lexical|"
        r"token|unknown|parse|information|reliably|hand)"
    ),
    re.IGNORECASE,
)
LATIN_WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")
ENGLISH_ZIPF_MIN = 3.5


def tokenize(text: str | float | None) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    return text.split()


def normalized_alpha_tokens(text: str | float | None) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    return [token.lower() for token in LATIN_WORD_RE.findall(text)]


def normalize_vocab_tokens(vocab: set[str] | None) -> set[str]:
    normalized: set[str] = set()
    for token in vocab or set():
        normalized.update(normalized_alpha_tokens(str(token)))
    return normalized


def build_english_lookup(
    token_lists: pd.Series, min_zipf: float = ENGLISH_ZIPF_MIN
) -> dict[str, bool]:
    unique_tokens = sorted(
        {token for tokens in token_lists for token in tokens if len(token) >= 2}
    )
    if zipf_frequency is None:
        return {token: False for token in unique_tokens}
    return {token: zipf_frequency(token, "en") >= min_zipf for token in unique_tokens}


def english_word_tokens(
    pred_alpha_tokens: list[str],
    target_vocab_alpha: set[str],
    english_lookup: dict[str, bool],
) -> list[str]:
    return [
        token
        for token in dict.fromkeys(pred_alpha_tokens)
        if english_lookup.get(token, False) and token not in target_vocab_alpha
    ]


def counter_overlap(left: list[str], right: list[str]) -> int:
    return sum((Counter(left) & Counter(right)).values())


def side_b_vocab(grammar: dict) -> set[str]:
    tokens: list[str] = []
    for key in [
        "verbs",
        "nouns",
        "propns",
        "prons",
        "adjs",
        "det_def",
        "det_indef",
        "comps",
    ]:
        tokens.extend(grammar["b"].get(key, []))
    for key in [
        "verb_paradigms",
        "noun_paradigms",
        "propn_paradigms",
        "pronoun_paradigms",
    ]:
        for item in grammar["b"].get(key, []):
            if not isinstance(item, dict):
                continue
            tokens.extend(str(item.get("lemma", "")).split())
            tokens.extend(str(item.get("form", "")).split())
            if "forms" in item:
                for form in item["forms"].values():
                    tokens.extend(str(form).split())
    vocab: set[str] = set()
    for token in tokens:
        vocab.update(str(token).split())
    return vocab


def build_vocab_map() -> dict[str, set[str]]:
    vocab_map: dict[str, set[str]] = {}
    for exp_dir in [
        "wordorder_exp",
        "wordorder_large_exp",
        "size_exp",
        "orthography_exp",
        "orthography_large_exp",
        "agreement_exp",
    ]:
        for path in sorted((DATA_DIR / exp_dir).glob("grammar_*.json")):
            grammar_name = path.stem.split("grammar_")[1]
            vocab_map[grammar_name] = side_b_vocab(json.loads(path.read_text()))
    return vocab_map


def classify_error_family(row: pd.Series) -> str:
    pred_tokens = row["pred_tokens"]
    ref_tokens = row["ref_tokens"]
    vocab = row["target_vocab"] if isinstance(row["target_vocab"], set) else set()
    pred_len = len(pred_tokens)
    ref_len = len(ref_tokens)

    if row["exact_match"]:
        return "exact"
    if row["failure_mode"] == "word_order_only":
        return "pure_reordering"
    if row["failure_mode"] in {"wrong_script", "diacritic_drop"}:
        return "script_or_form"
    if row["meta_response"]:
        return "task_escape"
    if (
        row["failure_mode"] in {"copied_source", "source_lexicon_intrusion"}
        or row["source_share"] >= 0.5
    ):
        return "source_leakage"
    if row["failure_mode"] == "repetition_loop":
        return "degeneration"
    if row["failure_mode"] in {"partial_span", "too_short"}:
        return "truncation_or_omission"
    if (
        pred_len < ref_len
        and row["target_precision"] >= 0.8
        and row["target_recall"] < 0.8
    ):
        return "truncation_or_omission"
    if (
        pred_len
        and vocab
        and row["licensed_token_share"] >= 0.8
        and row["target_recall"] >= 0.5
    ):
        return "licensed_near_miss"
    if pred_len and vocab and row["licensed_token_share"] < 0.8:
        return "unlicensed_or_hallucinated"
    return "other"


def family_share(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    counts = df.groupby(group_cols, observed=False).size().rename("count").reset_index()
    totals = (
        df.groupby(group_cols[:-1], observed=False).size().rename("total").reset_index()
    )
    merged = counts.merge(totals, on=group_cols[:-1], how="left")
    merged["share"] = merged["count"] / merged["total"]
    return merged


def pct(value: float) -> str:
    return f"{100 * value:.1f}%"


def family_label(family: str) -> str:
    return FAMILY_LABELS.get(family, family.replace("_", " "))


def family_label_list(families: list[str]) -> list[str]:
    return [family_label(family) for family in families]


vocab_map = build_vocab_map()
target_vocab_alpha_map = {
    grammar_name: normalize_vocab_tokens(vocab)
    for grammar_name, vocab in vocab_map.items()
}
rows_df = pd.read_csv(CACHE_DIR / "rows.csv")
if "dataset" not in rows_df.columns:
    rows_df["dataset"] = (
        rows_df["exp"]
        .map(
            {
                "wordorder": "wordorder_exp",
                "size": "size_exp",
                "agreement": "agreement_exp",
                "orthography": "orthography_exp",
            }
        )
        .fillna("unknown")
    )

cached_dataset_counts = (
    rows_df.groupby(["dataset", "exp"], observed=False)
    .size()
    .rename("rows")
    .reset_index()
)
rows_df = rows_df.loc[
    (
        (rows_df["exp"].astype(str) != "wordorder")
        | rows_df["dataset"].eq(WORD_ORDER_EXP)
    )
    & (
        (rows_df["exp"].astype(str) != "orthography")
        | rows_df["dataset"].eq(ORTHOGRAPHY_EXP)
    )
].copy()

rows_df["model_name"] = rows_df["fuzzy_model"]
rows_df["exp"] = pd.Categorical(
    rows_df["exp"], categories=EXPERIMENT_ORDER, ordered=True
)
rows_df["fuzzy_model"] = pd.Categorical(
    rows_df["fuzzy_model"], categories=MODEL_ORDER, ordered=True
)
rows_df["pred_tokens"] = rows_df["model_answer"].fillna("").map(tokenize)
rows_df["ref_tokens"] = rows_df["output_sentence"].fillna("").map(tokenize)
rows_df["src_tokens"] = rows_df["input_sentence"].fillna("").map(tokenize)
rows_df["pred_alpha_tokens"] = (
    rows_df["model_answer"].fillna("").map(normalized_alpha_tokens)
)
rows_df["target_vocab"] = rows_df["grammar_name"].map(vocab_map)
rows_df["target_vocab_alpha"] = rows_df["grammar_name"].map(target_vocab_alpha_map)
english_lookup = build_english_lookup(rows_df["pred_alpha_tokens"])
rows_df["english_word_tokens"] = rows_df.apply(
    lambda row: english_word_tokens(
        row["pred_alpha_tokens"],
        row["target_vocab_alpha"]
        if isinstance(row["target_vocab_alpha"], set)
        else set(),
        english_lookup,
    ),
    axis=1,
)
rows_df["contains_english_words"] = rows_df["english_word_tokens"].map(bool)
rows_df["english_word_count"] = rows_df["english_word_tokens"].map(len)
rows_df["target_recall"] = rows_df.apply(
    lambda row: 0.0
    if not row["ref_tokens"]
    else counter_overlap(row["pred_tokens"], row["ref_tokens"])
    / len(row["ref_tokens"]),
    axis=1,
)
rows_df["target_precision"] = rows_df.apply(
    lambda row: 0.0
    if not row["pred_tokens"]
    else counter_overlap(row["pred_tokens"], row["ref_tokens"])
    / len(row["pred_tokens"]),
    axis=1,
)
rows_df["source_share"] = rows_df.apply(
    lambda row: 0.0
    if not row["pred_tokens"]
    else counter_overlap(row["pred_tokens"], row["src_tokens"])
    / len(row["pred_tokens"]),
    axis=1,
)
rows_df["licensed_token_share"] = rows_df.apply(
    lambda row: 0.0
    if not row["pred_tokens"]
    else sum(
        token
        in (row["target_vocab"] if isinstance(row["target_vocab"], set) else set())
        for token in row["pred_tokens"]
    )
    / len(row["pred_tokens"]),
    axis=1,
)
rows_df["fully_licensed"] = rows_df["licensed_token_share"].eq(1.0)
rows_df["same_len"] = rows_df["pred_len"] == rows_df["ref_len"]
rows_df["meta_response"] = rows_df["model_answer"].fillna("").str.contains(META_RE)
rows_df["error_family"] = rows_df.apply(classify_error_family, axis=1)
wrong_df = rows_df.loc[~rows_df["exact_match"]].copy()
size_order = [
    f"{int(value):,}"
    for value in sorted(
        rows_df.loc[rows_df["exp"] == "size", "n_words"].dropna().unique()
    )
]
rows_df["size_bin"] = pd.Categorical(
    rows_df["n_words"].map(
        lambda value: f"{int(value):,}" if pd.notna(value) else np.nan
    ),
    categories=size_order,
    ordered=True,
)
wrong_df["size_bin"] = rows_df.loc[wrong_df.index, "size_bin"]

print(f"Using WORD_ORDER_EXP={WORD_ORDER_EXP}")
print(f"Using ORTHOGRAPHY_EXP={ORTHOGRAPHY_EXP}")
print(
    f"Loaded {len(rows_df):,} rows and {len(wrong_df):,} wrong answers "
    f"after dataset filtering."
)
print(
    "Error families:",
    ", ".join(
        family for family in FAMILY_ORDER if family in set(wrong_df["error_family"])
    ),
)
print(
    "Wrong answers with English lexical intrusion:",
    pct(wrong_df["contains_english_words"].mean()),
)
selected_counts = cached_dataset_counts.loc[
    cached_dataset_counts["dataset"].isin([WORD_ORDER_EXP, ORTHOGRAPHY_EXP])
].copy()
print(selected_counts.sort_values(["exp", "dataset"]).to_string(index=False))


## Plan

Hypotheses:
- Many rows that look like `hallucinated_vocab` in the first pass are actually *recall errors*: the model stays inside the target lexicon but retrieves the wrong lexical item or inflection.
- Weak-model failures on harder grammars should shift from subtle adequacy issues toward source leakage and explicit English task escape.
- `orthography` should isolate form/script problems from the rest of the translation pipeline.

Metrics used here:
- `target_recall` / `target_precision`: multiset token overlap with the reference.
- `source_share`: how much of the prediction is copied from the source sentence.
- `licensed_token_share`: how much of the prediction stays inside the grammar's target-side vocabulary.
- `meta_response`: whether the model answers in English about the task instead of translating.


Dataset selectors:
- `WORD_ORDER_EXP` and `ORTHOGRAPHY_EXP` at the top of the setup cell choose which cached dataset backs the `wordorder` and `orthography` analyses.


## A Compact Error Taxonomy

The first-pass labels are useful, but they over-separate some patterns and under-separate others. The family view below answers a more causal question: when the model is wrong, is it still behaving like a translator, or has it switched into some other failure mode?

Alongside the mutually exclusive taxonomy, I also track one overlapping diagnostic: **English lexical intrusion**. This fires when the model's final answer contains common English words that are *not* licensed by that grammar's target vocabulary, so accidental homographs in the target lexicon do not count.

In [ ]:
family_share_df = family_share(wrong_df, ["exp", "fuzzy_model", "error_family"])
dominant_family_df = (
    family_share_df.sort_values(
        ["exp", "fuzzy_model", "share"], ascending=[True, True, False]
    )
    .groupby(["exp", "fuzzy_model"], as_index=False, observed=False)
    .head(1)
    .rename(
        columns={
            "error_family": "dominant_wrong_family",
            "share": "dominant_wrong_share",
        }
    )
)

task_escape_df = family_share_df.loc[
    family_share_df["error_family"] == "task_escape", ["exp", "fuzzy_model", "share"]
].rename(columns={"share": "task_escape_within_wrong"})

english_intrusion_df = (
    wrong_df.groupby(["exp", "fuzzy_model"], observed=False)["contains_english_words"]
    .mean()
    .reset_index(name="english_words_within_wrong")
)

topline_df = (
    rows_df.groupby(["exp", "fuzzy_model"], observed=False)
    .agg(
        rows=("custom_id", "size"),
        exact_match=("exact_match", "mean"),
        bow_match=("bow_match", "mean"),
        mean_ref_len=("ref_len", "mean"),
    )
    .reset_index()
    .merge(dominant_family_df, on=["exp", "fuzzy_model"], how="left")
    .merge(task_escape_df, on=["exp", "fuzzy_model"], how="left")
    .merge(english_intrusion_df, on=["exp", "fuzzy_model"], how="left")
    .fillna({"task_escape_within_wrong": 0.0, "english_words_within_wrong": 0.0})
)

display(
    topline_df.assign(
        exact_match=lambda df: (100 * df["exact_match"]).round(1),
        bow_match=lambda df: (100 * df["bow_match"]).round(1),
        dominant_wrong_family=lambda df: df["dominant_wrong_family"].map(family_label),
        dominant_wrong_share=lambda df: (100 * df["dominant_wrong_share"]).round(1),
        task_escape_within_wrong=lambda df: (
            100 * df["task_escape_within_wrong"]
        ).round(1),
        english_words_within_wrong=lambda df: (
            100 * df["english_words_within_wrong"]
        ).round(1),
        mean_ref_len=lambda df: df["mean_ref_len"].round(1),
    )[
        [
            "exp",
            "fuzzy_model",
            "rows",
            "exact_match",
            "bow_match",
            "dominant_wrong_family",
            "dominant_wrong_share",
            "task_escape_within_wrong",
            "english_words_within_wrong",
            "mean_ref_len",
        ]
    ].sort_values(["exp", "fuzzy_model"])
)

In [ ]:
taxonomy_descriptions = {
    "pure_reordering": (
        "All of the right target tokens are present, but the sequence order is wrong."
    ),
    "licensed_near_miss": (
        "The model stays mostly inside the target lexicon and overlaps heavily "
        "with the reference, but recalls the wrong lexical item or form."
    ),
    "truncation_or_omission": (
        "The output is mostly target-side material, but it drops a span or stops "
        "too early."
    ),
    "source_leakage": (
        "Source-language words leak into the answer, or the source is copied outright."
    ),
    "script_or_form": (
        "The lexical mapping may be close, but the orthography, diacritics, or "
        "surface realization is wrong."
    ),
    "task_escape": (
        "The model stops translating and answers in English about the task or its "
        "uncertainty."
    ),
    "degeneration": (
        "The decoder falls into repetition or a locally collapsed pattern."
    ),
    "unlicensed_or_hallucinated": (
        "The prediction introduces tokens that are not licensed by the grammar's "
        "target vocabulary."
    ),
    "other": "Residual target-like errors that do not fit the cleaner families above.",
}

taxonomy_order = [
    family for family in FAMILY_ORDER if family in set(wrong_df["error_family"])
]
taxonomy_rows = []
for family in taxonomy_order:
    example = (
        wrong_df.loc[wrong_df["error_family"] == family]
        .sort_values(
            [
                "target_recall",
                "licensed_token_share",
                "source_share",
                "exp",
                "model_name",
                "depth",
                "n_words",
            ],
            ascending=[False, False, True, True, True, True, True],
            na_position="last",
        )
        .head(1)
    )
    if example.empty:
        continue
    row = example.iloc[0]
    taxonomy_rows.append(
        {
            "error_family": family,
            "description": taxonomy_descriptions[family],
            "exp": row["exp"],
            "model": row["model_name"],
            "failure_mode": row["failure_mode"],
            "input_sentence": row["input_sentence"],
            "output_sentence": row["output_sentence"],
            "model_answer": row["model_answer"],
        }
    )

glossary_sections = ["### Taxonomy Glossary: descriptions plus one example per family"]
for item in taxonomy_rows:
    glossary_sections.extend(
        [
            f"#### `{family_label(item['error_family'])}`",
            item["description"],
            "",
            f"Example: `{item['exp']}` | `{item['model']}` | `{item['failure_mode']}`",
            f"- Input: `{item['input_sentence']}`",
            f"- Target: `{item['output_sentence']}`",
            f"- Prediction: `{item['model_answer']}`",
            "",
        ]
    )

display(Markdown("\n".join(glossary_sections)))

In [ ]:
plot_df = family_share_df.copy()
present_families = [
    family for family in FAMILY_ORDER if family in set(plot_df["error_family"])
]
plot_df["error_family"] = pd.Categorical(
    plot_df["error_family"], categories=present_families, ordered=True
)

taxonomy_experiment_order = ["size", "wordorder", "agreement", "orthography"]
experiment_panels = [
    exp
    for exp in taxonomy_experiment_order
    if (wrong_df["exp"].astype(str) == exp).any()
]
panel_models = {
    exp: [
        model
        for model in MODEL_ORDER
        if (
            (wrong_df["exp"].astype(str) == exp)
            & (wrong_df["fuzzy_model"].astype(str) == model)
        ).any()
    ]
    for exp in experiment_panels
}

ncols = 2
nrows = int(np.ceil(len(experiment_panels) / ncols))
fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(1.5 * aes.COLM_PAPER_WIDTH_IN, aes.FIG_HEIGHT_DOUBLEROW_DIFFAXES_IN),
    sharex=True,
)
axes = np.atleast_1d(axes).reshape(nrows, ncols)

for panel_idx, exp in enumerate(experiment_panels):
    row_idx, col_idx = divmod(panel_idx, ncols)
    ax = axes[row_idx, col_idx]
    model_order = panel_models[exp]
    exp_plot_df = plot_df.loc[plot_df["exp"].astype(str) == exp].copy()
    pivot_df = (
        exp_plot_df.pivot_table(
            index="fuzzy_model",
            columns="error_family",
            values="share",
            observed=False,
            fill_value=0.0,
        )
        .reindex(model_order)
        .reindex(columns=present_families, fill_value=0.0)
    )
    bar_height = 0.9
    bar_sep = 0.08
    panel_height = float(len(model_order))
    total_group_height = (
        len(model_order) * bar_height + max(len(model_order) - 1, 0) * bar_sep
    )
    group_offset = 0.5 * (panel_height - total_group_height)
    y = (
        group_offset
        + 0.5 * bar_height
        + np.arange(len(model_order)) * (bar_height + bar_sep)
    )
    left = np.zeros(len(model_order))

    for family in present_families:
        widths = pivot_df[family].to_numpy(dtype=float)
        ax.barh(
            y,
            widths,
            left=left,
            height=bar_height,
            color=FAMILY_PALETTE[family],
            edgecolor="white",
            linewidth=0.7,
        )
        left = left + widths

    exp_label = {
        "wordorder": "Word Order",
        "size": "Size",
        "agreement": "Morphology",
        "orthography": "Orthography",
    }.get(exp, exp.title())
    ax.set_title(exp_label, loc="left", fontweight="normal", fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_ylim(panel_height, 0)
    ax.set_yticks(y)
    ax.set_xticks([0.0, 0.25, 0.5, 0.75, 1.0])
    if col_idx == 0:
        ax.set_yticklabels(model_order)
    else:
        ax.set_yticklabels([])
    ax.invert_yaxis()
    ax.tick_params(axis="y", length=0)
    ax.xaxis.set_major_formatter(aes.PCT_FORMATTER)
    ax.set_ylabel("")
    if row_idx != nrows - 1:
        ax.set_xlabel("")
        ax.xaxis.set_visible(False)
        ax.spines["bottom"].set_visible(False)
    else:
        ax.set_xlabel("Share of wrong answers", fontsize=9)
        xlabels = ax.get_xticklabels()
        # make the last x-axis label flush with the right edge
        if xlabels:
            xlabels[0].set_horizontalalignment("left")
            xlabels[-1].set_horizontalalignment("right")

    sns.despine(ax=ax, left=True, bottom=row_idx != nrows - 1)

for panel_idx in range(len(experiment_panels), nrows * ncols):
    row_idx, col_idx = divmod(panel_idx, ncols)
    axes[row_idx, col_idx].set_visible(False)

legend_handles = [
    plt.Rectangle(
        (0, 0), 1, 1, facecolor=FAMILY_PALETTE[family], edgecolor="white", linewidth=0.7
    )
    for family in present_families
]
legend_labels = family_label_list(present_families)
fig.legend(
    legend_handles,
    legend_labels,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=3,
    columnspacing=1.4,
    handletextpad=0.6,
    frameon=False,
)
fig.subplots_adjust(wspace=0, hspace=0.45)
plt.tight_layout()

aes.save_figure("error_taxonomy", fig=fig)


Two immediate takeaways:
- `agreement` is mostly not a hallucination problem. Even when the models fail, they often stay inside the target lexicon and produce structurally plausible target sentences.
- The weakest models on `wordorder`, `size`, and `orthography` frequently stop acting like translators at all: source vocab and English task escape become major mass.


In [ ]:
portrait_df = (
    wrong_df.groupby("error_family", observed=False)
    .agg(
        target_recall=("target_recall", "mean"),
        target_precision=("target_precision", "mean"),
        source_share=("source_share", "mean"),
        licensed_token_share=("licensed_token_share", "mean"),
        meta_response=("meta_response", "mean"),
        contains_english_words=("contains_english_words", "mean"),
    )
    .reindex(
        [family for family in FAMILY_ORDER if family in set(wrong_df["error_family"])]
    )
)

metric_labels = {
    "target_recall": "target recall",
    "target_precision": "target precision",
    "source_share": "source share",
    "licensed_token_share": "licensed share",
    "meta_response": "meta response",
    "contains_english_words": "english words",
}
portrait_display_df = (100 * portrait_df).round(1).rename(columns=metric_labels)
portrait_display_df.index = portrait_display_df.index.map(family_label)

display(portrait_display_df)

fig, ax = plt.subplots(figsize=FIGSIZE_HEATMAP)
portrait_heatmap_df = portrait_df.rename(index=FAMILY_LABELS, columns=metric_labels)
sns.heatmap(
    portrait_heatmap_df,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
    cbar_kws={"label": "Mean value"},
    ax=ax,
)
ax.set_title(
    "Error-family portraits: overlap, source copying, licensing, and English leakage"
)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## Difficulty Gradients

The next question is not just *what* the error families are, but *when* they appear. The plots below separate two different failure trajectories:
- a graceful degradation trajectory, where predictions remain target-like but drift lexically;
- a brittle trajectory, where the model leaks source words or abandons the task in English.


In [ ]:
wordorder_depth_df = family_share(
    wrong_df.loc[wrong_df["exp"].astype(str) == "wordorder"],
    ["fuzzy_model", "depth", "error_family"],
)
plot_families = [
    "licensed_near_miss",
    "pure_reordering",
    "source_leakage",
    "task_escape",
    "unlicensed_or_hallucinated",
]
wordorder_plot_df = wordorder_depth_df.loc[
    wordorder_depth_df["error_family"].isin(plot_families)
].copy()
wordorder_plot_df["share_pct"] = 100 * wordorder_plot_df["share"]
wordorder_plot_df["error_family_label"] = wordorder_plot_df["error_family"].map(
    family_label
)
plot_palette = {
    family_label(family): FAMILY_PALETTE[family] for family in plot_families
}
plot_order = family_label_list(plot_families)

plot_models = [
    model
    for model in MODEL_ORDER
    if model in set(wordorder_plot_df["fuzzy_model"].astype(str))
]

fig, axes = plt.subplots(
    1,
    len(plot_models),
    figsize=(
        max(1.0, len(plot_models) / 2) * aes.COLM_PAPER_WIDTH_IN,
        2 * aes.FIG_HEIGHT_SINGLE_ROW_IN,
    ),
    sharey=True,
)
axes = np.atleast_1d(axes)
for ax, model in zip(axes, plot_models):
    subset = wordorder_plot_df.loc[
        wordorder_plot_df["fuzzy_model"].astype(str) == model
    ]
    sns.lineplot(
        data=subset,
        x="depth",
        y="share_pct",
        hue="error_family_label",
        hue_order=plot_order,
        palette=plot_palette,
        marker="o",
        linewidth=2,
        ax=ax,
    )
    ax.set_title(aes.MODEL_DISPLAY_NAMES.get(model, model))
    ax.set_xlabel("Sentence depth")
    ax.set_ylabel("Share of wrong answers (%)")
    ax.set_ylim(0, 100)
    if ax is axes[-1]:
        ax.legend(title="", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    else:
        ax.get_legend().remove()

plt.tight_layout()
plt.show()

In [ ]:
size_family_df = family_share(
    wrong_df.loc[wrong_df["exp"].astype(str) == "size"],
    ["fuzzy_model", "size_bin", "error_family"],
)
plot_families = [
    "licensed_near_miss",
    "pure_reordering",
    "source_leakage",
    "task_escape",
    "unlicensed_or_hallucinated",
]
size_plot_df = size_family_df.loc[
    size_family_df["error_family"].isin(plot_families)
].copy()
size_plot_df["share_pct"] = 100 * size_plot_df["share"]
size_plot_df["error_family_label"] = size_plot_df["error_family"].map(family_label)
plot_palette = {
    family_label(family): FAMILY_PALETTE[family] for family in plot_families
}
plot_order = family_label_list(plot_families)

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE_THREE_PANEL, sharey=True)
for ax, model in zip(axes, MODEL_ORDER):
    subset = size_plot_df.loc[size_plot_df["fuzzy_model"].astype(str) == model]
    if subset.empty:
        ax.set_visible(False)
        continue
    sns.lineplot(
        data=subset,
        x="size_bin",
        y="share_pct",
        hue="error_family_label",
        hue_order=plot_order,
        palette=plot_palette,
        marker="o",
        linewidth=2,
        ax=ax,
    )
    ax.set_title(model)
    ax.set_xlabel("Target lexicon size (`n_words`)")
    ax.set_ylabel("Share of wrong answers (%)")
    ax.tick_params(axis="x", rotation=45)
    if ax is axes[0]:
        ax.legend(title="", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    else:
        ax.get_legend().remove()

fig.suptitle(
    "Grammar size changes the *kind* of failure, especially for smaller models", y=1.08
)
plt.tight_layout()
plt.show()


## Agreement: Well-Formed But Wrong

`agreement` is the clearest case where the family view changes the interpretation. The main question here is whether failures are random, or whether the model is producing a target-language sentence with the wrong feature binding.


In [ ]:
agreement_df = rows_df.loc[rows_df["exp"].astype(str) == "agreement"].copy()
agreement_summary_df = (
    agreement_df.groupby(["fuzzy_model", "agreement_condition"], observed=False)
    .apply(
        lambda group: pd.Series(
            {
                "rows": len(group),
                "exact_match": group["exact_match"].mean(),
                "fully_licensed_within_wrong": group.loc[
                    ~group["exact_match"], "fully_licensed"
                ].mean(),
                "licensed_same_len_within_wrong": (
                    group.loc[~group["exact_match"], "fully_licensed"]
                    & group.loc[~group["exact_match"], "same_len"]
                ).mean(),
                "licensed_near_miss_within_wrong": (
                    group.loc[~group["exact_match"], "error_family"]
                    == "licensed_near_miss"
                ).mean(),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

agreement_display_df = agreement_summary_df.copy()
for col in [
    "exact_match",
    "fully_licensed_within_wrong",
    "licensed_same_len_within_wrong",
    "licensed_near_miss_within_wrong",
]:
    agreement_display_df[col] = (100 * agreement_display_df[col]).round(1)
agreement_display_df = agreement_display_df.rename(
    columns={
        "exact_match": "exact match",
        "fully_licensed_within_wrong": "fully licensed within wrong",
        "licensed_same_len_within_wrong": "same-length licensed within wrong",
        "licensed_near_miss_within_wrong": "recall error within wrong",
    }
)

display(agreement_display_df.sort_values(["fuzzy_model", "agreement_condition"]))

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE_THREE_PANEL, sharey=False)
for ax, metric, title in zip(
    axes,
    ["exact_match", "fully_licensed_within_wrong", "licensed_near_miss_within_wrong"],
    [
        "Exact-match rate",
        "Wrong outputs that stay fully inside target vocab",
        "Wrong outputs that are recall errors",
    ],
):
    sns.barplot(
        data=agreement_summary_df,
        x="agreement_condition",
        y=metric,
        hue="fuzzy_model",
        palette=sns.color_palette("Reds", n_colors=3),
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Rate")
    ax.tick_params(axis="x", rotation=30)
    if ax is axes[0]:
        ax.legend(title="", frameon=False, loc="upper right")
    else:
        ax.get_legend().remove()

fig.suptitle(
    (
        "Agreement errors are usually target-side binding mistakes, "
        "not source copying or English escape"
    ),
    y=1.08,
)
plt.tight_layout()
plt.show()


Interpretation:
- Translating *into* agreement (`NoAgr -> Agr`) is the hardest setting in exact-match terms.
- But even there, many wrong answers are still fully licensed target strings, which suggests a feature-binding problem rather than a total breakdown.
- `NoAgr -> NoAgr` is the easiest condition and also the one where wrong answers are most often fully licensed and same-length.



## Orthography: Script Handling Versus Translation

`orthography` is useful because it cleanly separates lexical/structural translation from writing-system realization. The table and plots below show that the two models fail in qualitatively different ways.


In [ ]:
orth_df = rows_df.loc[rows_df["exp"].astype(str) == "orthography"].copy()
orth_summary_df = (
    orth_df.groupby(["fuzzy_model", "target_orthography"], observed=False)
    .apply(
        lambda group: pd.Series(
            {
                "rows": len(group),
                "exact_match": group["exact_match"].mean(),
                "script_or_form": (group["error_family"] == "script_or_form").mean(),
                "task_escape": (group["error_family"] == "task_escape").mean(),
                "source_leakage": (group["error_family"] == "source_leakage").mean(),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

orth_summary_plot_df = orth_summary_df.loc[
    orth_summary_df["fuzzy_model"].astype(str).isin(MODEL_ORDER)
].copy()
orth_summary_plot_df["fuzzy_model"] = orth_summary_plot_df["fuzzy_model"].astype(str)
orth_display_df = orth_summary_plot_df.copy()
for col in ["exact_match", "script_or_form", "task_escape", "source_leakage"]:
    orth_display_df[col] = (100 * orth_display_df[col]).round(1)
orth_display_df = orth_display_df.rename(
    columns={
        "exact_match": "exact match",
        "script_or_form": "orthography",
        "task_escape": "task escape",
        "source_leakage": "source vocab",
    }
)

display(orth_display_df.sort_values(["fuzzy_model", "target_orthography"]))

fig, axes = plt.subplots(
    1,
    3,
    figsize=(1.6 * aes.COLM_PAPER_WIDTH_IN, aes.FIG_HEIGHT_DOUBLE_ROW_IN),
    sharey=False,
)
for ax, metric, title in zip(
    axes,
    ["exact_match", "script_or_form", "task_escape"],
    ["Exact-match rate", "Orthography failure rate", "English task-escape rate"],
):
    sns.barplot(
        data=orth_summary_plot_df,
        x="target_orthography",
        y=metric,
        hue="fuzzy_model",
        hue_order=[
            model
            for model in MODEL_ORDER
            if model in set(orth_summary_plot_df["fuzzy_model"])
        ],
        palette=aes.PALETTE_MODELS,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Rate")
    ax.tick_params(axis="x", rotation=30)
    if ax is axes[0]:
        ax.legend(title="", frameon=False, loc="upper right")
    else:
        ax.get_legend().remove()

fig.suptitle(
    "Orthography splits script realization problems from broader translation failure",
    y=1.08,
)
plt.tight_layout()
plt.show()


## Literature Grounding

The taxonomy above is deliberately lightweight, but it tracks familiar distinctions from prior MT and generation work:

- **MQM** frames translation errors as multidimensional quality problems rather than a single scalar score, which is useful here because exact-match collapses near-miss and task-abandonment failures together. See the MQM overview and shared-task framing in [Lommel et al. (2014)](https://aclanthology.org/W14-3302/).
- **Adequacy versus target-side fluency**: [Tu et al. (2017), *Context Gates for Neural Machine Translation*](https://aclanthology.org/Q17-1007/) is a useful reminder that outputs can remain fluent while becoming weakly conditioned on the source. That is close to the `recall error` story here.
- **Hallucination / detachment from the source**: [Guerreiro et al. (2023), *Hallucinations in Neural Machine Translation*](https://aclanthology.org/2023.findings-acl.283/) is relevant for the `task_escape` and `hallucinated vocab` families.
- **Degeneration loops**: [Holtzman et al. (2020), *The Curious Case of Neural Text Degeneration*](https://openreview.net/forum?id=rygGQyrFvH) fits the `degeneration` family.

The distinctive result in this repository is that many failures are **not** classical hallucinations. Especially in `agreement`, the models often produce a target-language sentence that is almost right, but not semantically/feature-wise correct.


In [ ]:
EXAMPLE_COLUMNS = [
    "exp",
    "model_name",
    "failure_mode",
    "error_family",
    "depth",
    "n_words",
    "target_recall",
    "licensed_token_share",
    "source_share",
    "input_sentence",
    "output_sentence",
    "model_answer",
]
ENGLISH_EXAMPLE_COLUMNS = EXAMPLE_COLUMNS + ["english_word_tokens"]


def show_examples(family: str, n: int = 3, exp: str | None = None) -> pd.DataFrame:
    subset = wrong_df.loc[wrong_df["error_family"] == family].copy()
    if exp is not None:
        subset = subset.loc[subset["exp"].astype(str) == exp]
    subset = subset.sort_values(
        ["model_name", "exp", "depth", "n_words"], na_position="last"
    )
    subset = subset.head(n)
    display_df = subset[EXAMPLE_COLUMNS].copy()
    display_df["error_family"] = display_df["error_family"].map(family_label)
    for col in ["target_recall", "licensed_token_share", "source_share"]:
        display_df[col] = display_df[col].round(2)
    return display_df


def show_english_examples(
    n_per_family: int = 1, exp: str | None = None
) -> pd.DataFrame:
    subset = wrong_df.loc[wrong_df["contains_english_words"]].copy()
    if exp is not None:
        subset = subset.loc[subset["exp"].astype(str) == exp]
    subset = subset.sort_values(
        [
            "english_word_count",
            "target_recall",
            "model_name",
            "exp",
            "depth",
            "n_words",
        ],
        ascending=[False, False, True, True, True, True],
        na_position="last",
    )
    subset = (
        subset.groupby("error_family", observed=False, as_index=False)
        .head(n_per_family)
        .sort_values(["error_family", "model_name", "exp"], na_position="last")
    )
    display_df = subset[ENGLISH_EXAMPLE_COLUMNS].copy()
    display_df["error_family"] = display_df["error_family"].map(family_label)
    for col in ["target_recall", "licensed_token_share", "source_share"]:
        display_df[col] = display_df[col].round(2)
    return display_df


example_specs = [
    ("licensed_near_miss", "agreement", "Recall error examples (agreement)"),
    ("task_escape", "wordorder", "Task-escape examples (word order)"),
    ("source_leakage", None, "Source-vocab examples"),
    ("script_or_form", "orthography", "Orthography examples"),
    ("truncation_or_omission", None, "Omission examples"),
]

for family, exp, title in example_specs:
    display(Markdown(f"### {title}"))
    display(show_examples(family=family, exp=exp))

display(Markdown("### English-word intrusion examples across families"))
display(show_english_examples())

## Takeaways

- The most important correction to the first-pass story is that many errors are **target-language recall errors**, not generic hallucinations. This is strongest in `agreement`, where wrong outputs often stay fully inside the target lexicon.
- Difficulty changes the *kind* of error. Stronger models tend to degrade gracefully into recall errors or word ordering mistakes; weaker models increasingly leak source vocab or answer in English about the task.
- English lexical intrusion is broader than `task_escape`: some failures are explicit English refusals, while others are straight English translations or mixed English-target outputs.
- `orthography` is genuinely special: it reveals a separate axis of failure around script and diacritic realization, which would be easy to misread as ordinary lexical error if we only looked at exact match.
- For follow-up work, the most promising next step is to distinguish **lexical substitution**, **feature-binding**, and **scope/attachment** errors inside the large `recall error` bucket, especially for `agreement`.